In [1]:
import sys
import os
import importlib.util
import cv2
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import warnings
from skimage import io, transform
from skimage.util import img_as_ubyte

fusefilter_path = os.path.abspath("../defenselib/fuse_filter.py")

spec = importlib.util.spec_from_file_location("fuse_filter", fusefilter_path)
fusefilter = importlib.util.module_from_spec(spec)
sys.modules["fusefilter"] = fusefilter
spec.loader.exec_module(fusefilter)


compressdiff_path = os.path.abspath("../defenselib/spatial_heterogeneity.py")

spec = importlib.util.spec_from_file_location("compressdiff", compressdiff_path)
compressdiff = importlib.util.module_from_spec(spec)
sys.modules["compressdiff"] = compressdiff
spec.loader.exec_module(compressdiff)

In [2]:
ROOT_DIR = "C:\\Adrianov\\Projects\\Project-Satanael\\"
DEMO_DIR = os.path.join(ROOT_DIR, 'attack_demo')
APRICOT_TEST_DIR = os.path.join(ROOT_DIR, 'data', 'APRICOTv1.0', 'Images', 'Test')
TJUDHD_TEST_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random', 'images')

In [3]:
def save_heatmap(img, path, cmap='grey'):
    plt.imsave(path, img, cmap=cmap)

In [5]:
ratio_mi = 0.5 # ratio_cd = 1-ratio_mi
kernel_pram = 80
thresh_pram = 80 # percentile, from small to big

# filenames_combi = []

warnings.filterwarnings(
    "ignore",
    message="The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem."
)

i = len(filenames_combi)


for root, _, files in os.walk(TJUDHD_TEST_DIR):
    for file in files:
        if (file.lower() in filenames_combi):
            continue
        if file.lower().endswith('.jpg'):
            print(file)
            impath = os.path.join(TJUDHD_TEST_DIR, file)
            ori_img = Image.open(impath).convert('RGB')
            image = np.array(ori_img)
            ori_height, ori_width, _ = image.shape
        
            
            resized_image = image
            
            resized_image_uint8 = img_as_ubyte(resized_image)
            
            io.imsave(impath, resized_image_uint8)
            savefig_path = os.path.join(ROOT_DIR, 'results_segment_grey')

            mi_img, cd_img, fuse_img = fusefilter.fuse_heatmap(impath, ori_height, ori_width)
            if not os.path.exists(savefig_path):
                os.makedirs(savefig_path)
            
            threshold = np.percentile(fuse_img, thresh_pram)
            h_t, h_t_o, h_t_o_c, h_t_o_c_o = fusefilter.heatmap_filter(fuse_img, threshold, ori_height, ori_width)

            save_heatmap(mi_img, os.path.join(savefig_path, file + "_mi_heatmap.png"))
            save_heatmap(cd_img, os.path.join(savefig_path, file + "_cd_heatmap.png"))
            save_heatmap(fuse_img, os.path.join(savefig_path, file + "_fuse_heatmap.png"))
            save_heatmap(h_t, os.path.join(savefig_path, file + "_h_t.png"))
            save_heatmap(h_t_o, os.path.join(savefig_path, file + "_h_t_o.png"))
            save_heatmap(h_t_o_c, os.path.join(savefig_path, file + "_h_t_o_c.png"))
            save_heatmap(h_t_o_c_o, os.path.join(savefig_path, file + "_h_t_o_c_o.png"))
            i += 1
            print(f"File {file} saved. {i} heatmap(s) processed")
            filenames_combi.append(file)
            

1500014005934.jpg
(1199, 1623)
h_mi.shape (1199, 1623)
height , width 1200 1624
h_cd.shape (296, 402)
h_mi resize to ori size
h_cd resize to ori size
h_mi_max: 3.929252
h_mi_min: 0.0
h_cd_max: 0.7808051
h_cd_min: 0.00061082735
len(h_fuse) 1948800
15
File 1500014005934.jpg saved. 687 heatmap(s) processed
1500014070659.jpg
(1199, 1623)
h_mi.shape (1199, 1623)
height , width 1200 1624
h_cd.shape (296, 402)
h_mi resize to ori size
h_cd resize to ori size
h_mi_max: 3.9717858
h_mi_min: 0.0
h_cd_max: 0.7888037
h_cd_min: 0.0015515259
len(h_fuse) 1948800
15
File 1500014070659.jpg saved. 688 heatmap(s) processed
1500014083856.jpg
(1199, 1623)
h_mi.shape (1199, 1623)
height , width 1200 1624
h_cd.shape (296, 402)
h_mi resize to ori size
h_cd resize to ori size
h_mi_max: 3.847258
h_mi_min: 0.0
h_cd_max: 0.8210259
h_cd_min: 0.0003240404
len(h_fuse) 1948800
15
File 1500014083856.jpg saved. 689 heatmap(s) processed
1500014120470.jpg
(1199, 1623)
h_mi.shape (1199, 1623)
height , width 1200 1624
h_cd.s

In [4]:
import os
import re
results_dir = os.path.join(ROOT_DIR, "results_segment_grey")

# Get all file names in the directory
files = os.listdir(results_dir)

unique_bases = set()

for f in files:
    match = re.match(r"(.+?\.jpg)", f)  # everything up to and including .jpg
    if match:
        unique_bases.add(match.group(1))
        
filenames_combi = list(unique_bases)
print(filenames_combi)
print(len(filenames_combi))

['1499499092945.jpg', '1499497903521.jpg', '1499656517339.jpg', '1499478218571.jpg', '1499999435925.jpg', '1497494857672.jpg', '1499738034017.jpg', '1500002677969.jpg', '1499823658132.jpg', '1497219657784.jpg', '1499568775385.jpg', '1499481136509.jpg', '1499554377620.jpg', '1497246474255.jpg', '1499658095313.jpg', '1499560023738.jpg', '1497509781334.jpg', '1499473843153.jpg', '1499469185375.jpg', '1499559689414.jpg', '1497399601562.jpg', '1499812334697.jpg', '1499639769594.jpg', '1499561961729.jpg', '1499998531888.jpg', '1499556813096.jpg', '1499512878565.jpg', '1499738623420.jpg', '1497395011098.jpg', '1499473341363.jpg', '1499933571776.jpg', '1499502569302.jpg', '1499562833443.jpg', '1499998008663.jpg', '1500005407713.jpg', '1497409214088.jpg', '1499571056733.jpg', '1499600617138.jpg', '1499816623593.jpg', '1499995506605.jpg', '1499468734550.jpg', '1497517122441.jpg', '1499763664048.jpg', '1500004915796.jpg', '1499916916442.jpg', '1497397409493.jpg', '1499933298822.jpg', '14973384350

In [7]:
print(filenames_combi)

['1497182366270.jpg', '1497216533789.jpg', '1497216851965.jpg', '1497217395572.jpg', '1497217569982.jpg', '1497217990942.jpg']


In [4]:
import ast

string_data = """['frc10_10.jpg', 'frc10_11.jpg', 'frc10_12.jpg', 'frc10_13.jpg', 'frc10_14.jpg', 'frc10_15.jpg', 'frc10_16.jpg', 'frc10_4.jpg', 'frc10_6.jpg', 'frc10_7.jpg', 'frc10_9.jpg', 'frc1_0.jpg', 'frc1_1.jpg', 'frc1_10.jpg', 'frc1_11.jpg', 'frc1_12.jpg', 'frc1_14.jpg', 'frc1_15.jpg', 'frc1_16.jpg', 'frc1_18.jpg', 'frc1_19.jpg', 'frc1_2.jpg', 'frc1_20.jpg', 'frc1_21.jpg', 'frc1_22.jpg', 'frc1_23.jpg', 'frc1_3.jpg', 'frc1_5.jpg', 'frc1_6.jpg', 'frc1_7.jpg', 'frc1_8.jpg', 'frc1_9.jpg', 'frc2_0.jpg', 'frc2_1.jpg', 'frc2_10.jpg', 'frc2_11.jpg', 'frc2_12.jpg', 'frc2_13.jpg', 'frc2_14.jpg', 'frc2_2.jpg', 'frc2_3.jpg', 'frc2_4.jpg', 'frc2_5.jpg', 'frc2_6.jpg', 'frc2_7.jpg', 'frc2_8.jpg', 'frc2_9.jpg', 'frc3_11.jpg', 'frc3_12.jpg', 'frc3_13.jpg', 'frc3_14.jpg', 'frc3_3.jpg', 'frc3_4.jpg', 'frc3_5.jpg', 'frc3_6.jpg', 'frc3_7.jpg', 'frc3_8.jpg', 'frc3_9.jpg', 'frc4_0.jpg', 'frc4_1.jpg', 'frc4_10.jpg', 'frc4_11.jpg', 'frc4_12.jpg', 'frc4_13.jpg', 'frc4_19.jpg', 'frc4_2.jpg', 'frc4_20.jpg', 'frc4_21.jpg', 'frc4_22.jpg', 'frc4_23.jpg', 'frc4_24.jpg', 'frc4_25.jpg', 'frc4_26.jpg', 'frc4_27.jpg', 'frc4_28.jpg', 'frc4_29.jpg', 'frc4_3.jpg', 'frc4_30.jpg', 'frc4_31.jpg', 'frc4_32.jpg', 'frc4_33.jpg', 'frc4_34.jpg', 'frc4_35.jpg', 'frc4_36.jpg', 'frc4_37.jpg', 'frc4_38.jpg', 'frc4_39.jpg', 'frc4_4.jpg', 'frc4_40.jpg', 'frc4_41.jpg', 'frc4_42.jpg', 'frc4_43.jpg', 'frc4_44.jpg', 'frc4_45.jpg', 'frc4_46.jpg', 'frc4_5.jpg', 'frc4_6.jpg', 'frc4_7.jpg', 'frc4_8.jpg', 'frc4_9.jpg', 'frc5_0.jpg', 'frc5_1.jpg', 'frc5_10.jpg', 'frc5_11.jpg', 'frc5_12.jpg', 'frc5_14.jpg', 'frc5_15.jpg', 'frc5_16.jpg', 'frc5_17.jpg', 'frc5_18.jpg', 'frc5_19.jpg', 'frc5_2.jpg', 'frc5_20.jpg', 'frc5_21.jpg', 'frc5_22.jpg', 'frc5_23.jpg', 'frc5_3.jpg', 'frc5_4.jpg', 'frc5_5.jpg', 'frc5_6.jpg', 'frc5_7.jpg', 'frc5_9.jpg', 'frc6_0.jpg', 'frc6_1.jpg', 'frc6_11.jpg', 'frc6_12.jpg', 'frc6_13.jpg', 'frc6_14.jpg', 'frc6_2.jpg', 'frc6_3.jpg', 'frc6_4.jpg', 'frc6_6.jpg', 'frc6_7.jpg', 'frc7_0.jpg', 'frc7_1.jpg', 'frc7_10.jpg', 'frc7_11.jpg', 'frc7_12.jpg', 'frc7_13.jpg', 'frc7_14.jpg', 'frc7_2.jpg', 'frc7_3.jpg', 'frc7_4.jpg', 'frc7_5.jpg', 'frc7_6.jpg', 'frc7_7.jpg', 'frc7_8.jpg', 'frc7_9.jpg', 'frc8_0.jpg', 'frc8_1.jpg', 'frc8_10.jpg', 'frc8_12.jpg', 'frc8_14.jpg', 'frc8_2.jpg', 'frc8_4.jpg', 'frc8_6.jpg', 'frc8_7.jpg', 'frc8_8.jpg', 'frc9_0.jpg', 'frc9_1.jpg', 'frc9_10.jpg', 'frc9_11.jpg', 'frc9_12.jpg', 'frc9_13.jpg', 'frc9_15.jpg', 'frc9_16.jpg', 'frc9_17.jpg', 'frc9_18.jpg', 'frc9_19.jpg', 'frc9_2.jpg', 'frc9_20.jpg', 'frc9_21.jpg', 'frc9_22.jpg', 'frc9_3.jpg', 'frc9_4.jpg', 'frc9_5.jpg', 'frc9_6.jpg', 'frc9_7.jpg', 'frc9_8.jpg', 'frc9_9.jpg', 'frs1_0.jpg', 'frs1_1.jpg', 'frs1_12.jpg', 'frs1_13.jpg', 'frs1_14.jpg', 'frs1_2.jpg', 'frs1_3.jpg', 'frs1_4.jpg', 'frs1_5.jpg', 'frs1_6.jpg', 'frs1_7.jpg', 'frs1_8.jpg', 'frs1_9.jpg', 'frs2_1.jpg', 'frs2_10.jpg', 'frs2_11.jpg', 'frs2_12.jpg', 'frs2_13.jpg', 'frs2_14.jpg', 'frs2_15.jpg', 'frs2_2.jpg', 'frs2_4.jpg', 'frs2_5.jpg', 'frs2_7.jpg', 'frs2_8.jpg', 'frs3_0.jpg', 'frs3_1.jpg', 'frs3_10.jpg', 'frs3_11.jpg', 'frs3_12.jpg', 'frs3_13.jpg', 'frs3_14.jpg', 'frs3_2.jpg', 'frs3_3.jpg', 'frs3_4.jpg', 'frs3_5.jpg', 'frs3_6.jpg', 'frs3_7.jpg', 'frs3_8.jpg', 'frs3_9.jpg', 'frs5_0.jpg']"""

# Convert string to list
filenames_combi = ast.literal_eval(string_data)

# Now img_list is a regular Python list
print(len(filenames_combi))  # Number of images
print(filenames_combi[:5])   # First 5 elements


221
['frc10_10.jpg', 'frc10_11.jpg', 'frc10_12.jpg', 'frc10_13.jpg', 'frc10_14.jpg']


In [12]:
print(len(filenames_combi))  # Number of images
print(filenames_combi)   # First 5 elements


810
['frc10_10.jpg', 'frc10_11.jpg', 'frc10_12.jpg', 'frc10_13.jpg', 'frc10_14.jpg', 'frc10_15.jpg', 'frc10_16.jpg', 'frc10_4.jpg', 'frc10_6.jpg', 'frc10_7.jpg', 'frc10_9.jpg', 'frc1_0.jpg', 'frc1_1.jpg', 'frc1_10.jpg', 'frc1_11.jpg', 'frc1_12.jpg', 'frc1_14.jpg', 'frc1_15.jpg', 'frc1_16.jpg', 'frc1_18.jpg', 'frc1_19.jpg', 'frc1_2.jpg', 'frc1_20.jpg', 'frc1_21.jpg', 'frc1_22.jpg', 'frc1_23.jpg', 'frc1_3.jpg', 'frc1_5.jpg', 'frc1_6.jpg', 'frc1_7.jpg', 'frc1_8.jpg', 'frc1_9.jpg', 'frc2_0.jpg', 'frc2_1.jpg', 'frc2_10.jpg', 'frc2_11.jpg', 'frc2_12.jpg', 'frc2_13.jpg', 'frc2_14.jpg', 'frc2_2.jpg', 'frc2_3.jpg', 'frc2_4.jpg', 'frc2_5.jpg', 'frc2_6.jpg', 'frc2_7.jpg', 'frc2_8.jpg', 'frc2_9.jpg', 'frc3_11.jpg', 'frc3_12.jpg', 'frc3_13.jpg', 'frc3_14.jpg', 'frc3_3.jpg', 'frc3_4.jpg', 'frc3_5.jpg', 'frc3_6.jpg', 'frc3_7.jpg', 'frc3_8.jpg', 'frc3_9.jpg', 'frc4_0.jpg', 'frc4_1.jpg', 'frc4_10.jpg', 'frc4_11.jpg', 'frc4_12.jpg', 'frc4_13.jpg', 'frc4_19.jpg', 'frc4_2.jpg', 'frc4_20.jpg', 'frc4_21.jpg

## Segment Patched TJU-DHD Dataset with Compression Difference Only

In [6]:
import sys
import os
import importlib.util
import cv2

compressdiff_path = os.path.abspath("../defenselib/spatial_heterogeneity.py")

spec = importlib.util.spec_from_file_location("spatial_heterogeneity", compressdiff_path)
compressdiff = importlib.util.module_from_spec(spec)
sys.modules["compressdiff"] = compressdiff
spec.loader.exec_module(compressdiff)

In [ ]:
kernel_pram = 60

filenames_combi = []
TJUDHD_TRAIN_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random_bigger', 'images')
i = len(filenames_combi)


savefig_path = os.path.join(ROOT_DIR, 'results_cd_grey')
if not os.path.exists(savefig_path):
    os.makedirs(savefig_path)
    
for root, _, files in os.walk(TJUDHD_TRAIN_DIR):
    for file in files:
        if (file.lower() in filenames_combi):
            continue
        if file.lower().endswith('.jpg'):
            print(file)
            
            impath = os.path.join(TJUDHD_TRAIN_DIR, file)
            
            OutputMap, OutputX = compressdiff.img_heatmap_cd(impath)
            average_OutputMap = np.mean(OutputMap, axis=0)
            OutputMap_max = np.max(average_OutputMap)
            OutputMap_min = np.min(average_OutputMap)
            out_height = len(average_OutputMap)
            out_width = len(average_OutputMap[0])
            average_OutputMap = [int((average_OutputMap[i][j]-OutputMap_min)*255/(OutputMap_max-OutputMap_min)) for i in range(out_height) for j in range(out_width)]
            flatNumpyArray = np.array(average_OutputMap,dtype=np.uint8)
            
            # Convert the array to make a grayscale image
            grayImage = flatNumpyArray.reshape(out_height, out_width)
            img = cv2.imread(impath)
            ori_height, ori_width, _ = img.shape
            grayImage = cv2.resize(grayImage, (ori_width, ori_height)) 

            # Morphological processing
            base_kernel_size = int(min(ori_height, ori_width)/kernel_pram)
            kernel=np.ones((base_kernel_size*2,base_kernel_size*2),np.uint8)
            opened = cv2.morphologyEx(grayImage, cv2.MORPH_OPEN,kernel, iterations=1)
            kernel=np.ones((base_kernel_size,base_kernel_size),np.uint8)
            closed=cv2.morphologyEx(opened,cv2.MORPH_CLOSE,kernel, iterations=2)
            kernel=np.ones((base_kernel_size*3,base_kernel_size*3),np.uint8)
            opened2=cv2.morphologyEx(closed,cv2.MORPH_OPEN,kernel, iterations=2)
            
            _, thresh_map_adversarial = cv2.threshold(opened2, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            
            save_heatmap(grayImage, os.path.join(savefig_path, file + "_cd.png"))
            save_heatmap(opened, os.path.join(savefig_path, file + "_cd_o.png"))
            save_heatmap(closed, os.path.join(savefig_path, file + "_cd_o_c.png"))
            save_heatmap(opened2, os.path.join(savefig_path, file + "_cd_o_c_o.png"))
            save_heatmap(thresh_map_adversarial, os.path.join(savefig_path, file + "_cd_thresh.png"))
            i += 1
            print(f"File {file} saved. {i} heatmap(s) processed")
            filenames_combi.append(file)

1496728971354.jpg
height , width 1200 1624
File 1496728971354.jpg saved. 1 heatmap(s) processed
1496789537998.jpg
height , width 1200 1624
File 1496789537998.jpg saved. 2 heatmap(s) processed
1496793883119.jpg
height , width 1200 1624
File 1496793883119.jpg saved. 3 heatmap(s) processed
1496795772097.jpg
height , width 1200 1624
File 1496795772097.jpg saved. 4 heatmap(s) processed
1496803310299.jpg
height , width 1200 1624
File 1496803310299.jpg saved. 5 heatmap(s) processed
1496823270363.jpg
height , width 1200 1624
File 1496823270363.jpg saved. 6 heatmap(s) processed
1496830925210.jpg
height , width 1200 1624
File 1496830925210.jpg saved. 7 heatmap(s) processed
1496831236394.jpg
height , width 1200 1624
File 1496831236394.jpg saved. 8 heatmap(s) processed
1496841113093.jpg
height , width 1200 1624
File 1496841113093.jpg saved. 9 heatmap(s) processed
1496841138204.jpg
height , width 1200 1624
File 1496841138204.jpg saved. 10 heatmap(s) processed
1496841947868.jpg
height , width 1200 1

In [12]:
filenames_combi = []
TJUDHD_TRAIN_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random', 'images')
i = len(filenames_combi)

savefig_path = os.path.join(ROOT_DIR, 'results_cd_grey')
if not os.path.exists(savefig_path):
    os.makedirs(savefig_path)
    
for root, _, files in os.walk(TJUDHD_TRAIN_DIR):
    for file in files:
        if (file.lower() in filenames_combi):
            continue
        if file.lower().endswith('.jpg'):
            print(file)
            
            impath = os.path.join(TJUDHD_TRAIN_DIR, file)
            image = np.array(ori_img)
            ori_height, ori_width, _ = image.shape
            _, cd_img, _ = fusefilter.fuse_heatmap(impath, ori_height, ori_width)
            threshold = np.percentile(cd_img, thresh_pram)
            h_t, h_t_o, h_t_o_c, h_t_o_c_o = fusefilter.heatmap_filter(cd_img, threshold, ori_height, ori_width)
            save_heatmap(cd_img, os.path.join(savefig_path, file + "_cd_heatmap.png"))
            save_heatmap(h_t, os.path.join(savefig_path, file + "_h_t.png"))
            save_heatmap(h_t_o, os.path.join(savefig_path, file + "_h_t_o.png"))
            save_heatmap(h_t_o_c, os.path.join(savefig_path, file + "_h_t_o_c.png"))
            save_heatmap(h_t_o_c_o, os.path.join(savefig_path, file + "_h_t_o_c_o.png"))
            i += 1
            print(f"File {file} saved. {i} heatmap(s) processed")
            filenames_combi.append(file)

1497182366270.jpg
(1199, 1623)
h_mi.shape (1199, 1623)
height , width 1200 1624
h_cd.shape (296, 402)
h_mi resize to ori size
h_cd resize to ori size
h_mi_max: 3.7754745
h_mi_min: 0.0
h_cd_max: 0.78411734
h_cd_min: 0.001805511
len(h_fuse) 1948800
15
File 1497182366270.jpg saved. 1 heatmap(s) processed
1497216533789.jpg


KeyboardInterrupt: 